# Dynamic ORB Strategy
***
A strategy that uses the ORB strategy to determine which timeframe and position is best to be traded.

**Pipeline:** libraries & config → data import → shared backtest engine → paper replication
(Zarattini, Barbon & Aziz 2024) → 13k-configuration search → analysis of the winner → winner vs SPY.
Every strategy in this notebook runs through the **same engine**, so results are directly comparable.

## Libraries & configuration

In [11]:
import itertools
import math
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings("ignore", message="All-NaN slice encountered")   # sessions with an empty window
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="invalid value encountered")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# ── Shared settings (every cell reads these) ────────────────────────────────
DATA_DIR = Path("/Users/andrew/main/trading/data/etfs")
SYMBOLS = ("CPER", "DIA", "GLD", "IEF", "IWM", "QQQ", "SHY", "SLV", "SPY", "TLT", "USO")
BENCHMARK = "SPY"
EQUITY_ETFS = {"SPY", "QQQ", "IWM", "DIA"}
INITIAL_EQUITY = 25_000.0
TICK = 0.01
COMMISSION = 0.0035                    # $/share/side (IBKR Pro tiered entry level — verify current)
SLIPPAGE_TICKS = 1                     # adverse ticks on every stop/market fill (the paper assumed 0)
IEX_SWITCH = pd.Timestamp("2022-03-07")  # data source changes from pitrading to IEX-only volume

## Data importing

In [12]:
# ═════════════════════════════════════════════════════════════════════════════
# DATA IMPORT — ETF 1-minute OHLCV
#
# Loads each ETF's 1-minute CSV into a DataFrame indexed by timestamp
# (Eastern Time wall-clock, bars labelled by bar-open time).
#
# Output: raw_1m[symbol] = DataFrame[open, high, low, close, volume, source]
# ═════════════════════════════════════════════════════════════════════════════
REQUIRED = ["datetime", "open", "high", "low", "close", "volume"]


def load_etf_1m(path: Path) -> pd.DataFrame:
    """Read one ETF CSV → sorted, de-duplicated, DatetimeIndexed OHLCV frame."""
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, engine="pyarrow")
    df.columns = df.columns.str.lower()
    missing = set(REQUIRED) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("datetime").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    if "source" not in df.columns:
        df["source"] = "unknown"
    return df[["open", "high", "low", "close", "volume", "source"]]


raw_1m: dict[str, pd.DataFrame] = {sym: load_etf_1m(DATA_DIR / f"{sym}_clean_1min.csv") for sym in SYMBOLS}

display(pd.DataFrame({
    sym: {"bars": len(df), "first": df.index.min(), "last": df.index.max(),
          "sessions": df.index.normalize().nunique(),
          "iex_from": df.index[df["source"].eq("iex")].min()}
    for sym, df in raw_1m.items()
}).T)

,bars,first,last,sessions,iex_from
CPER,109059,2011-11-15 09:37:00,2026-09-21 15:59:00,3315,2022-03-07 09:30:00
DIA,2067170,2002-12-30 09:30:00,2026-09-21 15:58:00,5966,2022-03-07 09:30:00
GLD,1924936,2004-11-18 09:30:00,2026-09-21 15:59:00,5490,2022-03-07 09:30:00
IEF,1560831,2002-12-30 09:31:00,2026-09-21 15:57:00,5966,2022-03-07 09:30:00
IWM,2217669,2002-12-30 09:30:00,2026-09-21 15:57:00,5966,2022-03-07 09:30:00
QQQ,2245015,2002-12-30 09:30:00,2026-09-21 15:56:00,5968,2022-03-07 09:30:00
SHY,1523666,2002-12-30 09:30:00,2026-09-21 15:59:00,5966,2022-03-07 09:30:00
SLV,1752691,2006-04-28 09:32:00,2026-09-21 15:59:00,5128,2022-03-07 09:33:00
SPY,2250959,2002-12-30 09:30:00,2026-09-21 15:58:00,5968,2022-03-07 09:30:00
TLT,2005004,2002-12-30 09:31:00,2026-09-21 15:59:00,5966,2022-03-07 09:30:00


## Backtest engine
One engine shared by every strategy below. Fill rules are conservative and identical everywhere:
* **Entry:** a stop order fills at `max(level, bar open)`; a bar-close entry fills at that bar's close.
* **Stops** are rounded away from entry. A stop-order entry is checked for a stop on its own bar. A gap through
  a stop fills at the bar open. Breakeven and trailing stops use the best price up to the *previous* bar.
* **Targets** need a one-tick trade-through. The stop wins same-bar ties.
* **Time exit** at the close of the exit bar, or of the last bar that exists before it.
* **Costs:** commission per share per side, plus `slip_ticks` adverse ticks on every stop/market fill.
* **Ex-ante features:** everything a filter uses is known before the trade (daily features are shifted one session).

In [13]:
RTH_OPEN, N_BARS = 9 * 60 + 30, 390            # column j = bar opening at 09:30 + j minutes
OR_WINDOWS = (5, 10, 15, 30, 60)
LOOKBACK = 14
MIN_OR_COVERAGE = 0.80                         # share of OR minutes that must have a bar
MIN_SESSION_COVERAGE = 0.80
MAX_OPEN_DELAY = 2                             # first bar must open by 09:32
MIN_PRICE = 5.0
MIN_STOP_TICKS = 3
TOL = TICK / 4                                 # float32 comparison tolerance


def _first_true(mask: np.ndarray) -> np.ndarray:
    """Index of the first True per row, -1 if none."""
    i = mask.argmax(axis=1)
    return np.where(mask[np.arange(len(mask)), i], i, -1)


# ── 1. Session panels: rows = sessions, columns = minutes after 09:30 (NaN = missing bar) ──
def build_panel(df: pd.DataFrame) -> dict:
    minute = df.index.hour * 60 + df.index.minute - RTH_OPEN
    df = df.loc[(minute >= 0) & (minute < N_BARS)]
    px = df[["open", "high", "low", "close"]]
    df = df.loc[(px > 0).all(axis=1) & (df["high"] >= df["low"]) & np.isfinite(px).all(axis=1)]
    col = (df.index.hour * 60 + df.index.minute - RTH_OPEN).to_numpy()
    codes, dates = pd.factorize(df.index.normalize(), sort=True)

    def mat(values: np.ndarray) -> np.ndarray:
        m = np.full((len(dates), N_BARS), np.nan, dtype=np.float32)
        m[codes, col] = values
        return m

    O, H, L, C, V = (mat(df[k].to_numpy(np.float64)) for k in ("open", "high", "low", "close", "volume"))
    present = ~np.isnan(C)
    has_bar = present.any(axis=1)
    first = np.where(has_bar, present.argmax(axis=1), -1)
    last = np.where(has_bar, N_BARS - 1 - present[:, ::-1].argmax(axis=1), -1)
    coverage = present.sum(axis=1) / np.maximum(last + 1, 1)
    rows = np.arange(len(dates))
    daily = pd.DataFrame({
        "open": np.where(has_bar, O[rows, np.maximum(first, 0)], np.nan),
        "high": np.nanmax(H, axis=1), "low": np.nanmin(L, axis=1),
        "close": np.where(has_bar, C[rows, np.maximum(last, 0)], np.nan),
        "volume": np.nansum(V, axis=1),
    }, index=pd.DatetimeIndex(dates, name="date")).astype(float).round(6)
    daily["source"] = df["source"].astype(str).groupby(codes).first().to_numpy()
    session_ok = has_bar & (first <= MAX_OPEN_DELAY) & (coverage >= MIN_SESSION_COVERAGE)
    return dict(dates=daily.index, O=O, H=H, L=L, C=C, V=V, first=first, last=last,
                daily=daily, session_ok=session_ok)


# ── 2. Ex-ante daily features (all shifted: known before 09:30) ──
def daily_features(d: pd.DataFrame) -> pd.DataFrame:
    pc = d["close"].shift(1)
    tr = pd.concat([d["high"] - d["low"], (d["high"] - pc).abs(), (d["low"] - pc).abs()], axis=1).max(axis=1)
    atr14 = tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    atr50 = tr.ewm(alpha=1 / 50, adjust=False, min_periods=50).mean()
    rng = d["high"] - d["low"]
    f = pd.DataFrame(index=d.index)
    f["atr"] = atr14.shift(1)
    f["avg_vol"] = d["volume"].rolling(LOOKBACK, min_periods=LOOKBACK).mean().shift(1)
    f["gap_atr"] = (d["open"] - pc) / f["atr"]
    f["trend"] = np.sign(d["close"] - d["close"].rolling(20, min_periods=20).mean()).shift(1)
    f["atr_ratio"] = (atr14 / atr50).shift(1)
    f["nr7"] = (rng <= rng.rolling(7, min_periods=7).min()).shift(1, fill_value=False).astype(bool)
    f["open"] = d["open"]
    return f


# ── 3. Opening-range cache per window (known at 09:30 + n) ──
def or_cache(p: dict, n: int) -> dict:
    O, H, L, C, V, d = p["O"], p["H"], p["L"], p["C"], p["V"], p["daily"]
    rows = np.arange(len(d))
    present = ~np.isnan(C[:, :n])
    last_in = n - 1 - present[:, ::-1].argmax(axis=1)
    ok = p["session_ok"] & (present.mean(axis=1) >= MIN_OR_COVERAGE) & (p["last"] > n)
    o_open = O[rows, np.maximum(p["first"], 0)].astype(float)
    o_close = C[rows, last_in].astype(float)
    vol = pd.Series(np.nansum(V[:, :n], axis=1), index=d.index)
    # Relative volume vs the previous 14 valid sessions; the OR must include the 09:30 print
    # (~40 % of 5-min OR volume); computed within each data-source regime (IEX switch).
    rv_ok = ok & ~np.isnan(V[:, 0])
    base = (vol.where(rv_ok).groupby(d["source"].to_numpy())
            .transform(lambda s: s.dropna().rolling(LOOKBACK).mean().shift(1).reindex(s.index)))
    return dict(ok=ok,
                hi=np.nanmax(H[:, :n], axis=1).astype(float), lo=np.nanmin(L[:, :n], axis=1).astype(float),
                dir=np.where(ok, np.sign(np.round(o_close - o_open, 6)), 0).astype(int),
                rv=(vol / base).where(rv_ok & (base > 0)).to_numpy())


t0 = time.perf_counter()
panels = {sym: build_panel(df) for sym, df in raw_1m.items()}
FEAT = {s: daily_features(p["daily"]) for s, p in panels.items()}
ORC = {s: {n: or_cache(p, n) for n in OR_WINDOWS} for s, p in panels.items()}
CAL = pd.DatetimeIndex(sorted(set().union(*[set(p["dates"]) for p in panels.values()])))
YEARS = CAL.year.to_numpy()
YEAR_IDS = np.unique(YEARS)
YIDX = np.searchsorted(YEAR_IDS, YEARS)
R_SPY = panels[BENCHMARK]["daily"]["close"].reindex(CAL).ffill().pct_change().fillna(0.0).to_numpy()


# ── 4. Strategy definition ──
@dataclass(frozen=True)
class Logic:
    n: int                 # opening-range minutes
    dir_mode: str          # "candle" | "both" | "candle+gap" | "candle+trend"
    buffer: float          # entry beyond the OR, × ATR
    confirm: str           # "touch" (stop order) | "close" (bar-close confirmation)
    cutoff: int            # last entry column (minutes after 09:30)
    stop: tuple            # ("atr", k) | ("or_opp", 0) | ("or_mid", 0)
    be_R: float            # move stop to breakeven after +x R (0 = off)
    trail_atr: float       # trailing stop, × ATR from best price (0 = off)
    target_R: float        # profit target in R (0 = none)
    exit_col: int          # time exit column (389 = 15:59 bar → 16:00)

    def valid(self) -> bool:
        return self.cutoff > self.n and self.exit_col > self.n + 5 and not (self.be_R and self.trail_atr)


@dataclass(frozen=True)
class Filt:
    rv_min: float          # relative volume in the OR
    or_atr_max: float      # OR width / ATR
    gap_min: float         # |gap| / ATR
    regime: str            # "any" | "expanding" | "contracting" (ATR14 / ATR50)
    nr7: bool              # previous day was NR7
    universe: str          # "all" | "equity"
    top_n: int | None      # cross-sectional cap by relative volume
    min_avg_vol: float = 0.0   # paper liquidity filter (shares/day, previous 14 days)
    min_atr: float = 0.0       # paper volatility filter ($, ATR14 of previous days)


# ── 5. Vectorised simulator: one Logic × one asset, all sessions at once ──
#     Prices are mapped to "signed" space (shorts negated) so one code path serves both directions.
def simulate(sym: str, lg: Logic, slip_ticks: float = SLIPPAGE_TICKS) -> pd.DataFrame:
    p, f, oc = panels[sym], FEAT[sym], ORC[sym][lg.n]
    atr = f["atr"].to_numpy()
    base_ok = oc["ok"] & np.isfinite(atr) & (atr > 0) & (f["open"].to_numpy() > MIN_PRICE)
    cd = oc["dir"]
    if lg.dir_mode == "candle":
        dsel = cd
    elif lg.dir_mode == "candle+gap":
        dsel = np.where(np.sign(np.nan_to_num(f["gap_atr"].to_numpy())) == cd, cd, 0)
    elif lg.dir_mode == "candle+trend":
        dsel = np.where(np.nan_to_num(f["trend"].to_numpy()) == cd, cd, 0)
    else:
        dsel = np.where(base_ok, 2, 0)                      # "both": evaluate both sides
    rows = np.flatnonzero(base_ok & (dsel != 0))
    if rows.size == 0:
        return pd.DataFrame()

    n, slip = lg.n, slip_ticks * TICK
    O, H, L, C = (p[k][rows, n:lg.exit_col + 1] for k in ("O", "H", "L", "C"))
    j = np.arange(O.shape[1])[None, :]
    r_ = np.arange(len(rows))
    # last bar that actually exists at or before the time exit
    last_j = np.where(~np.isnan(C) & (j <= (p["last"][rows] - n)[:, None]), j, -1).max(axis=1)
    entry_lim = np.minimum(lg.cutoff - n, last_j - 1)
    a = atr[rows]
    hi, lo = oc["hi"][rows], oc["lo"][rows]
    trig_long = np.ceil(np.round((hi + lg.buffer * a) / TICK, 6)) * TICK
    trig_short = np.floor(np.round((lo - lg.buffer * a) / TICK, 6)) * TICK

    def scan(sign: float):
        ts = sign * (trig_long if sign > 0 else trig_short)
        Hs, Cs, Os = (H, C, O) if sign > 0 else (-L, -C, -O)
        win = j <= entry_lim[:, None]
        if lg.confirm == "touch":
            e = _first_true((Hs >= ts[:, None] - TOL) & win)
            fill = np.maximum(ts, np.round(Os[r_, np.maximum(e, 0)] / TICK) * TICK) + slip
        else:
            e = _first_true((Cs > ts[:, None] + TOL) & win)
            fill = np.round(Cs[r_, np.maximum(e, 0)] / TICK) * TICK + slip
        return e, fill

    if lg.dir_mode == "both":
        eL, fL = scan(1.0)
        eS, fS = scan(-1.0)
        eL_, eS_ = np.where(eL < 0, 10**6, eL), np.where(eS < 0, 10**6, eS)
        d = np.where(eL_ < eS_, 1.0, np.where(eS_ < eL_, -1.0, 0.0))     # same-bar double break → skip
        e = np.where(d > 0, eL, np.where(d < 0, eS, -1))
        fill_s = np.where(d > 0, fL, fS)
    else:
        d = dsel[rows].astype(float)
        e = np.full(len(rows), -1)
        fill_s = np.full(len(rows), np.nan)
        for sgn in (1.0, -1.0):
            mk = d == sgn
            if mk.any():
                ee, ff = scan(sgn)
                e, fill_s = np.where(mk, ee, e), np.where(mk, ff, fill_s)
    trig_ok = (e >= 0) & (d != 0)

    dd = d[:, None]
    Hs, Ls = np.where(dd > 0, H, -L), np.where(dd > 0, L, -H)
    Os, Cs = np.where(dd > 0, O, -O), np.where(dd > 0, C, -C)

    kind, k = lg.stop
    if kind == "atr":
        stop_s = fill_s - k * a
    elif kind == "or_opp":
        stop_s = np.where(d > 0, lo, -hi) - TICK
    else:
        stop_s = np.where(d > 0, (hi + lo) / 2, -(hi + lo) / 2)
    stop_s = np.floor(np.round(stop_s / TICK, 6)) * TICK      # rounded away from entry
    risk = fill_s - stop_s
    trig_ok &= np.isfinite(risk) & (risk >= MIN_STOP_TICKS * TICK)

    within = j <= last_j[:, None]
    ec = np.maximum(e, 0)[:, None]
    first_live = ec if lg.confirm == "touch" else ec + 1       # a stop-order entry bar can already stop out
    fav = np.where(within & (j >= first_live), Hs, -np.inf)
    run = np.maximum.accumulate(np.nan_to_num(fav, nan=-np.inf), axis=1)
    prev_best = np.maximum(np.concatenate([np.full((len(rows), 1), -np.inf), run[:, :-1]], axis=1),
                           fill_s[:, None])
    level = np.broadcast_to(stop_s[:, None], Hs.shape).copy()
    if lg.be_R > 0:
        level = np.where(prev_best - fill_s[:, None] >= lg.be_R * risk[:, None],
                         np.maximum(level, fill_s[:, None]), level)
    if lg.trail_atr > 0:
        level = np.maximum(level, prev_best - lg.trail_atr * a[:, None])

    js = _first_true(within & (j >= first_live) & (Ls <= level + TOL))
    lv = level[r_, np.maximum(js, 0)]
    o_s = np.round(Os[r_, np.maximum(js, 0)] / TICK) * TICK
    stop_fill = np.where(js == e, lv, np.minimum(lv, o_s)) - slip
    if lg.target_R > 0:
        tgt = np.ceil(np.round((fill_s + lg.target_R * risk) / TICK, 6)) * TICK
        jt = _first_true(within & (j >= ec + 1) & (Hs >= tgt[:, None] + TICK - TOL))
        tgt_fill = np.maximum(tgt, np.round(Os[r_, np.maximum(jt, 0)] / TICK) * TICK)
    else:
        jt, tgt_fill = np.full(len(rows), -1), np.full(len(rows), np.nan)
    big = 10**6
    js_, jt_ = np.where(js >= 0, js, big), np.where(jt >= 0, jt, big)
    by_stop = (js_ < big) & (js_ <= jt_)
    by_tgt = (jt_ < big) & (jt_ < js_)
    eod_fill = np.round(Cs[r_, np.maximum(last_j, 0)] / TICK) * TICK - slip
    exit_s = np.where(by_stop, stop_fill, np.where(by_tgt, tgt_fill, eod_fill))
    exit_j = np.where(by_stop, js, np.where(by_tgt, jt, last_j))

    pnl_ps = exit_s - fill_s - 2 * COMMISSION                   # $ per share after costs
    trig_ok &= np.isfinite(pnl_ps)
    fr = f.iloc[rows]
    out = pd.DataFrame({
        "asset": sym, "date": p["dates"][rows], "dir": d.astype(int), "triggered": trig_ok,
        "entry_col": e + n, "exit_col": exit_j + n, "entry": np.abs(fill_s), "risk_ps": risk,
        "pnl_ps": pnl_ps, "R": np.where(trig_ok, pnl_ps / np.where(risk > 0, risk, np.nan), np.nan),
        "rv": oc["rv"][rows], "or_atr": (hi - lo) / a, "gap_atr": fr["gap_atr"].to_numpy(),
        "atr_ratio": fr["atr_ratio"].to_numpy(), "nr7": fr["nr7"].to_numpy(),
        "atr": a, "avg_vol": fr["avg_vol"].to_numpy(),
        "reason": np.where(by_stop, "stop", np.where(by_tgt, "target", "time")),
    })
    out["cal_idx"] = CAL.get_indexer(out["date"])
    return out


def candidates(lg: Logic, slip_ticks: float = SLIPPAGE_TICKS) -> pd.DataFrame:
    parts = [c for c in (simulate(s, lg, slip_ticks) for s in panels) if not c.empty]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def apply_filter(c: pd.DataFrame, fl: Filt) -> pd.DataFrame:
    """OR-time filters + top-N ranking on candidates (before triggering is known) → triggered trades."""
    m = (c["or_atr"] <= fl.or_atr_max) & (c["gap_atr"].abs().fillna(0) >= fl.gap_min)
    if fl.rv_min > 0:
        m &= c["rv"] >= fl.rv_min
    if fl.regime == "expanding":
        m &= c["atr_ratio"] > 1
    elif fl.regime == "contracting":
        m &= c["atr_ratio"] < 1
    if fl.nr7:
        m &= c["nr7"]
    if fl.universe == "equity":
        m &= c["asset"].isin(EQUITY_ETFS)
    if fl.min_avg_vol > 0:
        m &= c["avg_vol"] >= fl.min_avg_vol
    if fl.min_atr > 0:
        m &= c["atr"] > fl.min_atr
    c = c.loc[m]
    if fl.top_n:
        c = c.loc[c.groupby("date")["rv"].rank(ascending=False, method="first", na_option="bottom") <= fl.top_n]
    return c.loc[c["triggered"]]


# ── 6. Portfolio sizing ──
def fixed_risk_returns(t: pd.DataFrame, risk: float = 0.0025, pos_cap: float = 1.0,
                       gross_cap: float = 4.0) -> np.ndarray:
    """Fast fractional sizing: `risk` of equity per trade, ≤ pos_cap × equity notional per trade,
    and the day's total notional scaled down to ≤ gross_cap × equity."""
    assert np.isfinite(t["R"].to_numpy()).all(), "non-finite trade R reached the portfolio"
    ent, rps, ci = t["entry"].to_numpy(), t["risk_ps"].to_numpy(), t["cal_idx"].to_numpy()
    w = np.minimum(1.0, pos_cap * rps / (risk * ent))                   # per-trade notional cap
    notional = risk * w * ent / rps                                     # × equity
    day_gross = np.bincount(ci, weights=notional, minlength=len(CAL))
    w = w * np.minimum(1.0, gross_cap / np.maximum(day_gross, 1e-12))[ci]
    return np.bincount(ci, weights=risk * w * t["R"].to_numpy(), minlength=len(CAL))


def run_risk(t: pd.DataFrame, rk: dict) -> np.ndarray:
    """Sequential, integer-share simulation with risk controls (keys: risk, gross_lev, max_trades,
    daily_loss, dd_throttle, eq_filter; optional pos_cap = max notional per trade × equity)."""
    t = t.sort_values(["cal_idx", "entry_col"])
    ci, ent, rps, pps = (t[k].to_numpy() for k in ("cal_idx", "entry", "risk_ps", "pnl_ps"))
    ecol, xcol = t["entry_col"].to_numpy(), t["exit_col"].to_numpy()
    pos_cap = rk.get("pos_cap", math.inf)
    day_pnl = np.zeros(len(CAL))
    equity = peak = INITIAL_EQUITY
    starts = np.flatnonzero(np.r_[True, ci[1:] != ci[:-1]]) if len(ci) else np.array([], int)
    ends = np.r_[starts[1:], len(ci)] if len(ci) else np.array([], int)
    last_day, eq_window = 0, []
    for s, e_ in zip(starts, ends):
        day = ci[s]
        eq_window = (eq_window + [equity] * (day - last_day))[-50:]   # equity carried over idle sessions
        last_day = day
        if equity <= 0:
            break
        if rk["eq_filter"] and len(eq_window) >= 50 and equity < np.mean(eq_window) * (1 - 1e-9):
            continue
        mult = 0.5 if rk["dd_throttle"] and equity < peak * (1 - rk["dd_throttle"]) else 1.0
        sod, taken, opened = equity, 0, []                         # opened: (exit_col, notional, pnl)
        for i in range(s, e_):
            if taken >= rk["max_trades"]:
                break
            if rk["daily_loss"] and sum(pn for xc, _, pn in opened if xc < ecol[i]) <= -rk["daily_loss"] * sod:
                break
            in_use = sum(nv for xc, nv, _ in opened if xc > ecol[i])
            q = math.floor(min(rk["risk"] * mult * sod / rps[i],
                               pos_cap * sod / ent[i],
                               max(rk["gross_lev"] * sod - in_use, 0.0) / ent[i]))
            if q <= 0:
                continue
            opened.append((xcol[i], q * ent[i], q * pps[i]))
            taken += 1
        day_pnl[day] = sum(pn for _, _, pn in opened)
        equity = sod + day_pnl[day]
        peak = max(peak, equity)
    eq = INITIAL_EQUITY + np.cumsum(day_pnl)
    prev = np.r_[INITIAL_EQUITY, eq[:-1]]
    return np.where(prev > 0, eq / prev - 1, 0.0)


# ── 7. Metrics ──
def summarize(r: np.ndarray, mask: np.ndarray) -> tuple[float, float, float]:
    """(Sharpe, CAGR, max drawdown) of daily returns r[mask]; fast path for the search."""
    x = r[mask]
    sd = x.std()
    g = np.r_[1.0, np.cumprod(1 + x)]
    sharpe = x.mean() / sd * np.sqrt(252) if sd > 0 else np.nan
    cagr = g[-1] ** (252 / len(x)) - 1 if g[-1] > 0 else -1.0
    return sharpe, cagr, (g / np.maximum.accumulate(g) - 1).min()


def perf_stats(r: pd.Series, bench: pd.Series, trades: pd.DataFrame | None = None) -> dict:
    """Full performance report for a daily return series (every session, zeros included)."""
    g = pd.concat([pd.Series([1.0], index=[r.index[0] - pd.Timedelta(days=1)]), (1 + r).cumprod()])
    dd = g / g.cummax() - 1
    trough = dd.idxmin()
    rec = g.loc[trough:][g.loc[trough:] >= g.loc[:trough].max()]
    monthly = (1 + r).resample("ME").prod() - 1
    yearly = (1 + r).resample("YE").prod() - 1
    beta = np.cov(r, bench)[0, 1] / bench.var() if bench.var() > 0 else np.nan
    down = np.sqrt(np.mean(np.minimum(r, 0) ** 2))
    cagr = g.iloc[-1] ** (252 / len(r)) - 1 if g.iloc[-1] > 0 else -1.0
    out = {
        "Total return %": 100 * (g.iloc[-1] - 1), "CAGR %": 100 * cagr,
        "Volatility %": 100 * r.std() * np.sqrt(252),
        "Sharpe": r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
        "Sortino": r.mean() / down * np.sqrt(252) if down > 0 else np.nan,
        "Max drawdown %": 100 * dd.min(), "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "Recovery (sessions)": len(r.loc[trough:rec.index[0]]) - 1 if len(rec) else np.nan,
        "Worst day %": 100 * r.min(), "Worst month %": 100 * monthly.min(),
        "Best year %": 100 * yearly.max(), "Worst year %": 100 * yearly.min(),
        "Up months %": 100 * (monthly > 0).mean(),
        "Corr. to SPY": r.corr(bench), "Beta to SPY": beta,
        "Alpha % (ann.)": 100 * (r.mean() - beta * bench.mean()) * 252,
    }
    if trades is not None:
        out["Trades"] = len(trades)
        out["Hit ratio %"] = 100 * (trades["R"] > 0).mean() if len(trades) else np.nan
    return out


print(f"Engine ready: {len(panels)} assets, {len(CAL):,} sessions "
      f"({CAL[0].date()} → {CAL[-1].date()}), built in {time.perf_counter() - t0:.0f}s")

Engine ready: 11 assets, 5,969 sessions (2002-12-30 → 2026-09-21), built in 4s


## Strategy importing
Imports the strategy directly from the research paper: Zarattini, Barbon & Aziz (2024), SSRN 4729284.

In [14]:
# ═════════════════════════════════════════════════════════════════════════════
# ZARATTINI, BARBON & AZIZ (2024) — paper replication on the ETF universe
#
#   • Universe filter (ex-ante): open > $5, avg volume of previous 14 days ≥ 1M shares, ATR(14) > $0.50.
#   • Direction: sign of the first n-minute candle; doji → no trade. Stop-order entry at the OR high/low.
#   • Stop 10 % of ATR(14) from entry; otherwise exit at the close. No target.
#   • Sizing: each position gets equity / N slots; a stop-out loses 1 % of that slot; 4× leverage per slot.
#   • "Stocks in Play": Relative Volume ≥ 100 %, top 20 by Relative Volume. Windows 5/15/30/60; COMBO = equal weight.
#   • Costs as in the paper: $0.0035/share, zero slippage. A 1-tick-slippage row shows cost sensitivity.
# ═════════════════════════════════════════════════════════════════════════════
PAPER_PERIOD = (pd.Timestamp("2016-01-01"), pd.Timestamp("2023-12-31"))
PAPER_SLOTS = len(panels)
PAPER_SIZING = dict(risk=0.01 / PAPER_SLOTS, pos_cap=4.0 / PAPER_SLOTS, gross_lev=4.0, max_trades=99,
                    daily_loss=0.0, dd_throttle=0.0, eq_filter=False)


def paper_logic(n: int) -> Logic:
    return Logic(n, "candle", 0.0, "touch", N_BARS - 1, ("atr", 0.10), 0.0, 0.0, 0.0, N_BARS - 1)


PAPER_BASE = Filt(0.0, math.inf, 0.0, "any", False, "all", None, min_avg_vol=1_000_000, min_atr=0.50)
PAPER_RV = Filt(1.0, math.inf, 0.0, "any", False, "all", 20, min_avg_vol=1_000_000, min_atr=0.50)

paper_cands = {n: candidates(paper_logic(n), slip_ticks=0) for n in (5, 15, 30, 60)}
paper_trades = {"5m ORB Base": apply_filter(paper_cands[5], PAPER_BASE)}
for n in (5, 15, 30, 60):
    paper_trades[f"{n}m ORB + Rel Vol"] = apply_filter(paper_cands[n], PAPER_RV)
paper_rets = pd.DataFrame({k: run_risk(t, PAPER_SIZING) for k, t in paper_trades.items()}, index=CAL)
paper_rets["COMBO (5/15/30/60 equal-weight)"] = paper_rets[[f"{n}m ORB + Rel Vol" for n in (5, 15, 30, 60)]].mean(axis=1)
paper_rets[f"{BENCHMARK} buy & hold"] = R_SPY
slip_test = apply_filter(candidates(paper_logic(5), slip_ticks=1), PAPER_RV)

print("Paper filters — share of tradable sessions passing each filter, by data-source era:")
filt_rows = []
for s, p in panels.items():
    f, ok = FEAT[s], p["session_ok"]
    for era, m in (("pre-IEX", ok & (p["dates"] < IEX_SWITCH)), ("IEX-only", ok & (p["dates"] >= IEX_SWITCH))):
        if m.any():
            filt_rows.append({"asset": s, "era": era, "sessions": int(m.sum()),
                              "avg vol ≥ 1M": (f["avg_vol"][m] >= 1_000_000).mean(),
                              "ATR > $0.50": (f["atr"][m] > 0.50).mean()})
display(pd.DataFrame(filt_rows).set_index(["asset", "era"]).round(2).unstack("era"))

bench = pd.Series(R_SPY, index=CAL)
for label, (lo, hi) in (("Full sample", (CAL[0], CAL[-1])), ("Paper period 2016–2023", PAPER_PERIOD)):
    m = (CAL >= lo) & (CAL <= hi)
    table = {k: perf_stats(paper_rets.loc[m, k], bench[m],
                           paper_trades[k].loc[paper_trades[k]["date"].between(lo, hi)] if k in paper_trades else None)
             for k in paper_rets}
    table["5m + Rel Vol, 1-tick slippage"] = perf_stats(pd.Series(run_risk(slip_test, PAPER_SIZING), index=CAL)[m],
                                                        bench[m], slip_test.loc[slip_test["date"].between(lo, hi)])
    print(f"\n{label}: {lo.date()} → {hi.date()}")
    display(pd.DataFrame(table).T[["Total return %", "CAGR %", "Volatility %", "Sharpe", "Hit ratio %",
                                   "Max drawdown %", "Worst day %", "Alpha % (ann.)", "Beta to SPY", "Trades"]].round(2))

styles = {"5m ORB Base": dict(color="#555555", width=1.5), "5m ORB + Rel Vol": dict(color="#1f77b4", width=2.5),
          "15m ORB + Rel Vol": dict(color="#2ca02c", width=1.2), "30m ORB + Rel Vol": dict(color="#9467bd", width=1.2),
          "60m ORB + Rel Vol": dict(color="#8c564b", width=1.2),
          "COMBO (5/15/30/60 equal-weight)": dict(color="#000000", width=2, dash="dot"),
          f"{BENCHMARK} buy & hold": dict(color="#d62728", width=3)}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04,
                    subplot_titles=("Equity ($, log scale)", "Drawdown (%)"))
for name in paper_rets:
    eq = INITIAL_EQUITY * (1 + paper_rets[name]).cumprod()
    fig.add_trace(go.Scatter(x=CAL, y=eq, name=name, line=styles[name], legendgroup=name), 1, 1)
    fig.add_trace(go.Scatter(x=CAL, y=100 * (eq / eq.cummax() - 1), name=name, showlegend=False,
                             line={**styles[name], "width": 1}, legendgroup=name), 2, 1)
for row in (1, 2):
    fig.add_vrect(x0=PAPER_PERIOD[0], x1=PAPER_PERIOD[1], fillcolor="#f2c94c", opacity=0.10, line_width=0, row=row, col=1)
fig.add_vline(x=IEX_SWITCH, line_dash="dash", line_color="grey", row=1, col=1)
fig.add_annotation(x=IEX_SWITCH, y=1, yref="y domain", text="data → IEX-only", showarrow=False,
                   xanchor="left", font=dict(size=10, color="grey"), row=1, col=1)
fig.add_annotation(x=PAPER_PERIOD[0], y=1, yref="y domain", text="paper period 2016–2023", showarrow=False,
                   xanchor="left", font=dict(size=10), row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_layout(title=(f"Zarattini, Barbon & Aziz (2024) n-minute ORB on {len(panels)} ETFs — benchmark: {BENCHMARK} buy & hold"
                         f"<br><sup>Stop 10 % ATR · EoD exit · 1 % risk per slot (equity / {PAPER_SLOTS}) · 4× leverage · "
                         f"${COMMISSION}/share · zero slippage (as in the paper)</sup>"),
                  template="plotly_white", height=820, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="left", x=0))
fig.show()

Paper filters — share of tradable sessions passing each filter, by data-source era:


sessions         avg vol ≥ 1M         ATR > $0.50        
era   IEX-only pre-IEX     IEX-only pre-IEX    IEX-only pre-IEX
asset                                                          
CPER       2.0    33.0         0.00    0.00        1.00    0.03
DIA      169.0  4614.0         0.01    1.00        1.00    1.00
GLD      458.0  3982.0         0.03    1.00        1.00    0.98
IEF       53.0  2147.0         0.02    0.82        0.62    0.28
IWM     1056.0  4370.0         0.16    1.00        1.00    1.00
QQQ     1078.0  4540.0         0.05    1.00        1.00    0.88
SHY        NaN  2536.0          NaN    0.73         NaN    0.00
SLV      322.0  3504.0         0.34    1.00        0.92    0.28
SPY     1095.0  4588.0         0.75    1.00        1.00    1.00
TLT     1003.0  3543.0         0.49    1.00        1.00    0.98
USO       78.0  3671.0         0.04    0.71        1.00    1.00


Full sample: 2002-12-30 → 2026-09-21


,Total return %,CAGR %,Volatility %,Sharpe,Hit ratio %,Max drawdown %,Worst day %,Alpha % (ann.),Beta to SPY,Trades
5m ORB Base,-34.94,-1.80,7.29,-0.21,16.90,-39.58,-1.81,-1.31,-0.02,26138.0
5m ORB + Rel Vol,-9.47,-0.42,4.27,-0.08,15.87,-19.56,-1.28,-0.20,-0.01,10002.0
15m ORB + Rel Vol,-23.89,-1.15,4.09,-0.26,16.47,-29.07,-1.57,-1.04,-0.00,9893.0
30m ORB + Rel Vol,-16.82,-0.77,3.72,-0.19,17.12,-27.02,-0.71,-0.67,-0.00,9894.0
60m ORB + Rel Vol,-17.42,-0.80,3.52,-0.21,19.13,-28.42,-0.72,-0.70,-0.00,9078.0
COMBO (5/15/30/60 equal-weight),-16.30,-0.75,2.75,-0.26,NaN,-21.30,-0.89,-0.65,-0.01,NaN
SPY buy & hold,1158.97,11.29,18.51,0.67,NaN,-55.20,-11.63,-0.00,1.00,NaN
"5m + Rel Vol, 1-tick slippage",-64.89,-4.32,3.96,-1.10,14.29,-66.46,-1.26,-4.26,-0.01,10002.0



Paper period 2016–2023: 2016-01-01 → 2023-12-31


,Total return %,CAGR %,Volatility %,Sharpe,Hit ratio %,Max drawdown %,Worst day %,Alpha % (ann.),Beta to SPY,Trades
5m ORB Base,-8.99,-1.17,6.44,-0.15,17.63,-21.63,-1.81,-0.90,-0.01,9758.0
5m ORB + Rel Vol,-0.68,-0.08,4.09,-0.00,16.45,-12.39,-1.28,-0.11,0.01,3690.0
15m ORB + Rel Vol,0.66,0.08,4.08,0.04,17.80,-15.78,-1.57,0.01,0.01,3568.0
30m ORB + Rel Vol,3.02,0.37,3.52,0.12,18.38,-8.85,-0.52,0.35,0.01,3514.0
60m ORB + Rel Vol,-2.73,-0.35,3.22,-0.09,19.60,-9.95,-0.55,-0.39,0.01,3239.0
COMBO (5/15/30/60 equal-weight),0.34,0.04,2.57,0.03,NaN,-7.08,-0.89,-0.03,0.01,NaN
SPY buy & hold,152.36,12.29,18.65,0.72,NaN,-33.85,-11.63,-0.00,1.00,NaN
"5m + Rel Vol, 1-tick slippage",-21.58,-3.00,3.93,-0.76,15.20,-28.19,-1.26,-3.10,0.01,3690.0


## Bootstrapping
This will help determine what the best strategy variation and parameters are.

In [15]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB VARIATION SEARCH — 13,000+ configurations + risk-management overlays
#
# Research basis for the search space (verify details against the sources):
#   • Zarattini, Barbon & Aziz (2024); Zarattini & Aziz (2023): 5-min ORB, ATR stops, EoD exit, Rel Vol.
#   • Tsai et al. (2019): short opening ranges work best on index products.
#   • Crabel (1990) / Fisher (2002, ACD): entry buffer beyond the range; NR7 days precede expansion.
#   • Holmberg et al. (2013); Lundström (2017): ORB profits rise with volatility.
#   • Wu et al. (2021): profit targets hurt; protective / trailing stops help.
#   • Gao, Han, Li & Zhou (2018, JFE) "Market intraday momentum": time-of-day effects.
#   • Practitioner risk controls: fixed-fractional risk, leverage cap, max trades/day, daily loss limit,
#     drawdown throttle, equity-curve filter.
#
# Protocol: selection uses 2003–2015 only (2016+ held out). Stage 1 = ~12,000 signal × exit × filter configs
# (fast fractional sizing, 4× gross cap). Stage 2 = ~1,400 risk overlays on the top 10 (integer shares).
# Reports: Deflated Sharpe, IS→OOS rank correlation, marginal effects, and walk-forward re-selection
# (pick the best config on all prior years, trade it the next year). Runtime ≈ 3 min; QUICK = True ≈ 30 s.
# ═════════════════════════════════════════════════════════════════════════════
QUICK = False
SEED = 7
N_LOGIC = 60 if QUICK else 500
FILTERS_PER_LOGIC = 6 if QUICK else 24
STAGE2_TOP_K = 3 if QUICK else 10
IS_END = pd.Timestamp("2015-12-31")
IS_MASK = CAL <= IS_END
MIN_IS_TRADES = 300
WF_MIN_TRAIN_YEARS = 5
WF_MIN_TRADES_PER_YEAR = 25
MAX_IS_DD = 0.25
STAGE1_SIZING = dict(risk=0.0025, pos_cap=1.0, gross_cap=4.0)

LOGIC_GRID = {
    "n":         [5, 10, 15, 30, 60],
    "dir_mode":  ["candle", "both", "candle+gap", "candle+trend"],
    "buffer":    [0.0, 0.02, 0.05],
    "confirm":   ["touch", "close"],
    "cutoff":    [60, 120, 389],                 # no new entries after 10:30 / 11:30 / close
    "stop":      [("atr", 0.05), ("atr", 0.10), ("atr", 0.15), ("atr", 0.25), ("atr", 0.50),
                  ("or_opp", 0.0), ("or_mid", 0.0)],
    "be_R":      [0.0, 1.0, 2.0],
    "trail_atr": [0.0, 0.25, 0.50],
    "target_R":  [0.0, 2.0, 5.0, 10.0],
    "exit_col":  [150, 270, 389],                # close of the 12:00 / 14:00 / 15:59 bar
}
FILTER_GRID = {
    "rv_min":     [0.0, 1.0, 1.5, 2.0, 3.0],
    "or_atr_max": [math.inf, 0.5, 0.3],
    "gap_min":    [0.0, 0.3],
    "regime":     ["any", "expanding", "contracting"],
    "nr7":        [False, True],
    "universe":   ["all", "equity"],
    "top_n":      [None, 3],
}
RISK_GRID = {
    "risk":        [0.0025, 0.005, 0.01],
    "gross_lev":   [2.0, 4.0],
    "max_trades":  [1, 3, 99],
    "daily_loss":  [0.0, 0.01],
    "dd_throttle": [0.0, 0.15],
    "eq_filter":   [False, True],
}
LOGIC_COLS = list(LOGIC_GRID)
FILTER_COLS = list(FILTER_GRID)

# ── Stage 1 ──────────────────────────────────────────────────────────────────
t_start = time.perf_counter()
rng = np.random.default_rng(SEED)
all_logics = [lg for lg in (Logic(*c) for c in itertools.product(*LOGIC_GRID.values())) if lg.valid()]
all_filters = [Filt(*c) for c in itertools.product(*FILTER_GRID.values())]
logic_pick = [all_logics[i] for i in rng.choice(len(all_logics), N_LOGIC, replace=False)]
print(f"Search space: {len(all_logics):,} logics × {len(all_filters):,} filters "
      f"→ sampling {N_LOGIC * FILTERS_PER_LOGIC:,} Stage-1 configs")

rows_out, year_stats = [], []                 # year_stats: per config (Σr, Σr², trades) per year
for li, lg in enumerate(logic_pick):
    cands = candidates(lg)
    if cands.empty:
        continue
    for fl in [all_filters[i] for i in rng.choice(len(all_filters), FILTERS_PER_LOGIC, replace=False)]:
        t = apply_filter(cands, fl)
        r = fixed_risk_returns(t, **STAGE1_SIZING)
        is_t = int((t["date"] <= IS_END).sum())
        s_is, s_oos = summarize(r, IS_MASK), summarize(r, ~IS_MASK)
        rows_out.append({**asdict(lg), "stop": f"{lg.stop[0]}:{lg.stop[1]}",
                         **{k: getattr(fl, k) for k in FILTER_COLS},
                         "is_trades": is_t, "oos_trades": len(t) - is_t,
                         "is_avgR": t.loc[t["date"] <= IS_END, "R"].mean(),
                         "oos_avgR": t.loc[t["date"] > IS_END, "R"].mean(),
                         "is_sharpe": s_is[0], "is_cagr": s_is[1], "is_mdd": s_is[2],
                         "oos_sharpe": s_oos[0], "oos_cagr": s_oos[1], "oos_mdd": s_oos[2],
                         "is_daily_sr": r[IS_MASK].mean() / r[IS_MASK].std() if r[IS_MASK].std() > 0 else np.nan,
                         "_logic": lg, "_filt": fl})
        year_stats.append(np.stack([np.bincount(YIDX, weights=r, minlength=len(YEAR_IDS)),
                                    np.bincount(YIDX, weights=r * r, minlength=len(YEAR_IDS)),
                                    np.bincount(YIDX[t["cal_idx"].to_numpy()], minlength=len(YEAR_IDS))]))
    if (li + 1) % max(1, N_LOGIC // 5) == 0:
        print(f"  {li + 1}/{N_LOGIC} logics  ({time.perf_counter() - t_start:.0f}s)")

res = pd.DataFrame(rows_out)
YS = np.stack(year_stats)
elig = res.loc[(res["is_trades"] >= MIN_IS_TRADES) & res["is_sharpe"].notna()]
if elig.empty:
    raise RuntimeError("No eligible configurations — lower MIN_IS_TRADES.")
rho, p_rho = stats.spearmanr(elig["is_sharpe"], elig["oos_sharpe"], nan_policy="omit")
top_dec = elig.nlargest(max(1, len(elig) // 10), "is_sharpe")
print(f"\nStage 1: {len(res):,} configs, {len(elig):,} with ≥ {MIN_IS_TRADES} IS trades. "
      f"IS→OOS rank correlation ρ = {rho:+.3f}; median OOS Sharpe: all {elig['oos_sharpe'].median():+.2f}, "
      f"top IS decile {top_dec['oos_sharpe'].median():+.2f}; OOS Sharpe > 0 for {100 * (elig['oos_sharpe'] > 0).mean():.0f} %")

show = LOGIC_COLS + FILTER_COLS + ["is_trades", "is_avgR", "is_sharpe", "oos_trades", "oos_avgR", "oos_sharpe",
                                   "oos_cagr", "oos_mdd"]
print("\nTop 15 Stage-1 configs by IN-SAMPLE Sharpe (selection never sees OOS columns):")
display(elig.nlargest(15, "is_sharpe")[show].round(3))

print("\nMarginal effects — median Sharpe of eligible configs per parameter value (OOS = 2016+):")
marg = []
for col in LOGIC_COLS + FILTER_COLS:
    key = elig[col].astype(str)
    g = elig.groupby(key)[["is_sharpe", "oos_sharpe"]].median()
    g["configs"] = key.value_counts()
    marg.append(g.assign(param=col).reset_index(names="value"))
display(pd.concat(marg).set_index(["param", "value"]).round(3))

# ── Stage 2: risk overlays ──────────────────────────────────────────────────
risk_grid = [dict(zip(RISK_GRID, c)) for c in itertools.product(*RISK_GRID.values())]
stage2_seeds = elig.sort_values("is_sharpe", ascending=False).drop_duplicates(subset=LOGIC_COLS).head(STAGE2_TOP_K)
s2_rows, s2_returns = [], {}
for sid, row in stage2_seeds.iterrows():
    trades = apply_filter(candidates(row["_logic"]), row["_filt"])
    for ri, rk in enumerate(risk_grid):
        r = run_risk(trades, rk)
        s_is, s_oos = summarize(r, IS_MASK), summarize(r, ~IS_MASK)
        s2_rows.append({"seed": sid, "ri": ri, **rk, "is_sharpe": s_is[0], "is_cagr": s_is[1], "is_mdd": s_is[2],
                        "is_calmar": s_is[1] / abs(s_is[2]) if s_is[2] < 0 else np.nan,
                        "oos_sharpe": s_oos[0], "oos_cagr": s_oos[1], "oos_mdd": s_oos[2],
                        "is_daily_sr": r[IS_MASK].mean() / r[IS_MASK].std() if r[IS_MASK].std() > 0 else np.nan})
        s2_returns[(sid, ri)] = r
s2 = pd.DataFrame(s2_rows)
s2_ok = s2.loc[s2["is_mdd"] >= -MAX_IS_DD].dropna(subset=["is_calmar"])
s2_ok = s2_ok if not s2_ok.empty else s2.dropna(subset=["is_calmar"])
best2 = s2_ok.loc[s2_ok["is_calmar"].idxmax()]
print(f"\nStage 2: {len(stage2_seeds)} configs × {len(risk_grid)} risk protocols = {len(s2):,} runs. "
      f"Selection: highest IS Calmar with IS MDD ≤ {MAX_IS_DD:.0%}. Top 10:")
display(s2_ok.nlargest(10, "is_calmar").drop(columns=["is_daily_sr", "ri"]).round(3))
print("Marginal effect of each risk control (median over Stage-2 runs):")
display(pd.concat([s2.groupby(c)[["is_sharpe", "is_mdd", "oos_sharpe", "oos_mdd"]].median().assign(param=c)
                   .reset_index(names="value") for c in RISK_GRID]).set_index(["param", "value"]).round(3))


# ── Honest estimates: Deflated Sharpe & walk-forward re-selection ───────────
def deflated_sharpe(r: np.ndarray, trial_srs: np.ndarray, n_trials: int) -> dict:
    sr = r.mean() / r.std()
    g = 0.5772156649
    sr0 = np.sqrt(np.nanvar(trial_srs, ddof=1)) * ((1 - g) * stats.norm.ppf(1 - 1 / n_trials)
                                                    + g * stats.norm.ppf(1 - 1 / (n_trials * np.e)))
    den = np.sqrt(max(1 - stats.skew(r) * sr + (stats.kurtosis(r, fisher=False) - 1) / 4 * sr ** 2, 1e-12))
    return {"DSR": stats.norm.cdf((sr - sr0) * np.sqrt(len(r) - 1) / den),
            "PSR (SR > 0)": stats.norm.cdf(sr * np.sqrt(len(r) - 1) / den), "trials": n_trials}


best1 = elig.loc[elig["is_sharpe"].idxmax()]
r_best1 = fixed_risk_returns(apply_filter(candidates(best1["_logic"]), best1["_filt"]), **STAGE1_SIZING)
r_best2 = s2_returns[(best2["seed"], int(best2["ri"]))]
n_trials = len(res) + len(s2)
trial_srs = np.r_[elig["is_daily_sr"].to_numpy(), s2["is_daily_sr"].to_numpy()]   # configs with meaningful Sharpe
print(f"\nDeflated Sharpe Ratio of the IS winners after {n_trials:,} trials "
      "(conservative here: much of the dispersion between configs is real, not noise):")
display(pd.DataFrame({"Stage-1 winner": deflated_sharpe(r_best1[IS_MASK], trial_srs, n_trials),
                      "Stage-2 winner": deflated_sharpe(r_best2[IS_MASK], trial_srs, n_trials)}).T)

wf_ret = np.zeros(len(CAL))
wf_log, wf_cache = [], {}
for yi, yr in enumerate(YEAR_IDS):
    if yi < WF_MIN_TRAIN_YEARS:
        continue
    s1_, s2_, ntr = YS[:, 0, :yi].sum(1), YS[:, 1, :yi].sum(1), YS[:, 2, :yi].sum(1)
    ndays = (YIDX < yi).sum()
    mu = s1_ / ndays
    score = np.where(ntr >= WF_MIN_TRADES_PER_YEAR * yi, mu / np.sqrt(np.maximum(s2_ / ndays - mu ** 2, 1e-18)), -np.inf)
    k = int(np.argmax(score))
    if k not in wf_cache:
        wf_cache[k] = fixed_risk_returns(apply_filter(candidates(res.iloc[k]["_logic"]), res.iloc[k]["_filt"]),
                                         **STAGE1_SIZING)
    m = YEARS == yr
    wf_ret[m] = wf_cache[k][m]
    wf_log.append({"trade_year": yr, "chosen_config": k, "train_sharpe": score[k] * np.sqrt(252),
                   "year_return_%": 100 * (np.prod(1 + wf_cache[k][m]) - 1)})
wf_log = pd.DataFrame(wf_log)
WF_MASK = np.isin(YEARS, wf_log["trade_year"])
print("\nWalk-forward re-selection log (config chosen using only prior years):")
display(wf_log.round(3))

series = {"Stage-1 IS winner": (r_best1, np.ones(len(CAL), bool)),
          "Stage-2 IS winner (with risk mgmt)": (r_best2, np.ones(len(CAL), bool)),
          "Walk-forward re-selection": (wf_ret, WF_MASK),
          f"{BENCHMARK} buy & hold": (R_SPY, np.ones(len(CAL), bool))}
summary = {}
for name, (r, active) in series.items():
    for label, mk in (("IS ≤ 2015", IS_MASK & active), ("OOS 2016+", ~IS_MASK & active)):
        if mk.any():
            sh, cg, md = summarize(r, mk)
            summary[(name, label)] = {"Sharpe": sh, "CAGR %": 100 * cg, "MDD %": 100 * md}
print("\nHeadline comparison (walk-forward rows only count the years it actually traded):")
display(pd.DataFrame(summary).T.round(2))
print("Stage-1 IS winner:", best1["_logic"], best1["_filt"], sep="\n  ")
print("Stage-2 risk protocol:", {k: (best2[k].item() if hasattr(best2[k], "item") else best2[k]) for k in RISK_GRID})
print(f"Search runtime {time.perf_counter() - t_start:.0f}s")

colors = {"Stage-1 IS winner": "#1f77b4", "Stage-2 IS winner (with risk mgmt)": "#2ca02c",
          "Walk-forward re-selection": "#9467bd", f"{BENCHMARK} buy & hold": "#d62728"}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04,
                    subplot_titles=("Equity ($, log scale)", "Drawdown (%)"))
for name, (r, _) in series.items():
    eq = INITIAL_EQUITY * np.cumprod(1 + r)
    w = 3 if name.startswith(BENCHMARK) else 2
    fig.add_trace(go.Scatter(x=CAL, y=eq, name=name, line=dict(color=colors[name], width=w), legendgroup=name), 1, 1)
    fig.add_trace(go.Scatter(x=CAL, y=100 * (eq / np.maximum.accumulate(eq) - 1), name=name, showlegend=False,
                             line=dict(color=colors[name], width=1), legendgroup=name), 2, 1)
for row in (1, 2):
    fig.add_vrect(x0=IS_END, x1=CAL[-1], fillcolor="grey", opacity=0.08, line_width=0, row=row, col=1)
fig.add_annotation(x=IS_END, y=1, yref="y domain", text="  out-of-sample (2016+)", showarrow=False,
                   xanchor="left", row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_layout(title=f"ORB variation search — {n_trials:,} configurations, selected on 2003–2015 only",
                  template="plotly_white", height=820, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="left", x=0))
fig.show()

fig2 = go.Figure(go.Scattergl(x=elig["is_sharpe"], y=elig["oos_sharpe"], mode="markers",
                              marker=dict(size=4, opacity=0.35, color=elig["n"], colorscale="Viridis",
                                          colorbar=dict(title="OR min")),
                              text=elig["dir_mode"] + " · " + elig["stop"], name="configs"))
lim = [np.nanmin(elig[["is_sharpe", "oos_sharpe"]].to_numpy()), np.nanmax(elig[["is_sharpe", "oos_sharpe"]].to_numpy())]
fig2.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(dash="dash", color="grey"), name="IS = OOS"))
fig2.update_layout(title=f"Overfitting map — IS vs OOS Sharpe for {len(elig):,} configs (ρ = {rho:+.2f})",
                   xaxis_title="In-sample Sharpe (2003–2015)", yaxis_title="Out-of-sample Sharpe (2016+)",
                   template="plotly_white", height=520)
fig2.show()

Search space: 141,120 logics × 720 filters → sampling 12,000 Stage-1 configs
  100/500 logics  (36s)
  200/500 logics  (70s)
  300/500 logics  (102s)
  400/500 logics  (134s)
  500/500 logics  (166s)

Stage 1: 12,000 configs, 4,221 with ≥ 300 IS trades. IS→OOS rank correlation ρ = +0.611; median OOS Sharpe: all -0.09, top IS decile +0.35; OOS Sharpe > 0 for 44 %

Top 15 Stage-1 configs by IN-SAMPLE Sharpe (selection never sees OOS columns):


,n,dir_mode,buffer,confirm,cutoff,stop,be_R,trail_atr,target_R,exit_col,rv_min,or_atr_max,gap_min,regime,nr7,universe,top_n,is_trades,is_avgR,is_sharpe,oos_trades,oos_avgR,oos_sharpe,oos_cagr,oos_mdd
9665,10,candle+gap,0.00,close,60,atr:0.5,0.0,0.5,0.0,389,0.0,0.5,0.3,any,False,all,NaN,3044,0.084,1.161,3393,0.009,0.305,0.013,-0.089
1093,15,candle+gap,0.00,close,60,atr:0.5,0.0,0.0,10.0,389,1.0,0.5,0.3,expanding,False,all,3.0,629,0.222,1.129,677,-0.022,-0.067,-0.002,-0.070
9323,15,candle+gap,0.02,close,120,or_mid:0.0,2.0,0.0,0.0,270,0.0,0.5,0.3,any,False,equity,NaN,1425,0.211,1.081,1419,0.243,1.183,0.068,-0.061
4702,15,candle+gap,0.05,touch,60,or_opp:0.0,1.0,0.0,0.0,389,0.0,0.5,0.3,expanding,False,equity,3.0,535,0.328,1.014,586,0.143,0.668,0.019,-0.064
4530,10,candle+gap,0.05,touch,120,atr:0.25,0.0,0.5,0.0,389,0.0,0.3,0.3,any,False,all,NaN,2730,0.131,0.984,2804,0.020,0.281,0.016,-0.142
1081,15,candle+gap,0.00,close,60,atr:0.5,0.0,0.0,10.0,389,1.5,inf,0.3,any,False,all,NaN,669,0.173,0.984,815,0.021,0.158,0.005,-0.051
1914,15,candle+gap,0.02,close,120,or_mid:0.0,1.0,0.0,5.0,270,0.0,0.3,0.3,any,False,equity,3.0,880,0.199,0.975,830,0.264,1.223,0.045,-0.038
2032,15,candle+gap,0.02,touch,389,atr:0.5,1.0,0.0,0.0,389,1.0,0.5,0.3,any,False,equity,NaN,1049,0.161,0.972,927,0.071,0.476,0.015,-0.054
2449,30,candle+gap,0.00,close,120,atr:0.25,0.0,0.0,5.0,389,1.0,0.3,0.3,any,False,all,NaN,585,0.242,0.971,531,0.024,0.075,0.001,-0.051
4698,15,candle+gap,0.05,touch,60,or_opp:0.0,1.0,0.0,0.0,389,1.0,0.5,0.3,any,False,all,NaN,1490,0.164,0.961,1585,0.021,0.248,0.010,-0.122



Marginal effects — median Sharpe of eligible configs per parameter value (OOS = 2016+):


is_sharpe  oos_sharpe  configs
param      value                                       
n          10               -0.564      -0.205      921
           15               -0.338       0.054     1007
           30               -0.414       0.008      763
           5                -0.636      -0.268     1068
           60               -0.421      -0.054      462
dir_mode   both             -0.772      -0.231     1186
           candle           -0.497      -0.102     1289
           candle+gap       -0.111       0.160      828
           candle+trend     -0.503      -0.215      918
buffer     0.0              -0.450      -0.112     1415
           0.02             -0.485      -0.091     1318
           0.05             -0.536      -0.064     1488
confirm    close            -0.302       0.061     1952
           touch            -0.684      -0.244     2269
cutoff     120              -0.440      -0.051     1462
           389              -0.484      -0.113     1519
           60               -0.565      -0.111     1240
stop       atr:0.05         -2.438      -1.113      612
           atr:0.1          -1.020      -0.290      611
           atr:0.15         -0.508      -0.077      622
           atr:0.25         -0.194       0.119      634
           atr:0.5          -0.197       0.074      616
           or_mid:0.0       -0.305       0.011      583
           or_opp:0.0       -0.245       0.087      543
be_R       0.0              -0.511      -0.079     2569
           1.0              -0.467      -0.133      876
           2.0              -0.416      -0.096      776
trail_atr  0.0              -0.402      -0.078     2396
           0.25             -0.775      -0.227      887
           0.5              -0.402      -0.016      938
target_R   0.0              -0.361       0.024     1224
           10.0             -0.563      -0.140     1014
           2.0              -0.703      -0.254      927
           5.0              -0.419      -0.062     1056
exit_col   150              -0.668      -0.164     1337
           270              -0.444      -0.085     1485
           389              -0.349      -0.027     1399
rv_min     0.0              -0.458      -0.026     1977
           1.0              -0.507      -0.068     1289
           1.5              -0.568      -0.174      657
           2.0              -0.432      -0.294      291
           3.0              -0.436      -1.090        7
or_atr_max 0.3              -0.423      -0.126     1113
           0.5              -0.510      -0.111     1500
           inf              -0.509      -0.043     1608
gap_min    0.0              -0.653      -0.103     2378
           0.3              -0.281      -0.078     1843
regime     any              -0.505      -0.054     1713
           contracting      -0.495      -0.098     1339
           expanding        -0.452      -0.125     1169
nr7        False            -0.613      -0.120     3306
           True             -0.258      -0.005      915
universe   all              -0.501      -0.318     2524
           equity           -0.468       0.295     1697
top_n      3.0              -0.487      -0.085     2129
           nan              -0.483      -0.100     2092


Stage 2: 10 configs × 144 risk protocols = 1,440 runs. Selection: highest IS Calmar with IS MDD ≤ 25%. Top 10:


,seed,risk,gross_lev,max_trades,daily_loss,dd_throttle,eq_filter,is_sharpe,is_cagr,is_mdd,is_calmar,oos_sharpe,oos_cagr,oos_mdd
321,9323,0.002,4.0,3,0.00,0.00,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
323,9323,0.002,4.0,3,0.00,0.15,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
325,9323,0.002,4.0,3,0.01,0.00,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
327,9323,0.002,4.0,3,0.01,0.15,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
329,9323,0.002,4.0,99,0.00,0.00,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
331,9323,0.002,4.0,99,0.00,0.15,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
333,9323,0.002,4.0,99,0.01,0.00,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
335,9323,0.002,4.0,99,0.01,0.15,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
377,9323,0.005,4.0,99,0.00,0.00,True,0.991,0.063,-0.054,1.165,0.324,0.014,-0.079
379,9323,0.005,4.0,99,0.00,0.15,True,0.991,0.063,-0.054,1.165,0.324,0.014,-0.079


... (5 outputs left out)

,trade_year,chosen_config,train_sharpe,year_return_%
0,2007,40,0.773,-0.486
1,2008,1093,1.086,2.082
2,2009,9665,1.213,7.411
3,2010,9665,1.256,3.758
4,2011,9665,1.149,7.198
5,2012,9665,1.181,-1.525
6,2013,1093,1.157,1.347
7,2014,1093,1.094,6.909
8,2015,1093,1.229,-0.087
9,2016,9665,1.161,-5.287



Headline comparison (walk-forward rows only count the years it actually traded):


Sharpe  CAGR %  MDD %
Stage-1 IS winner                  IS ≤ 2015    1.16    5.30  -6.09
                                   OOS 2016+    0.31    1.35  -8.86
Stage-2 IS winner (with risk mgmt) IS ≤ 2015    1.21    4.49  -3.40
                                   OOS 2016+    0.44    1.22  -3.89
Walk-forward re-selection          IS ≤ 2015    0.75    2.91  -6.09
                                   OOS 2016+    1.03    5.77  -6.85
SPY buy & hold                     IS ≤ 2015    0.54    8.77 -55.20
                                   OOS 2016+    0.85   14.42 -33.85

Stage-1 IS winner:
  Logic(n=10, dir_mode='candle+gap', buffer=0.0, confirm='close', cutoff=60, stop=('atr', 0.5), be_R=0.0, trail_atr=0.5, target_R=0.0, exit_col=389)
  Filt(rv_min=0.0, or_atr_max=0.5, gap_min=0.3, regime='any', nr7=False, universe='all', top_n=None, min_avg_vol=0.0, min_atr=0.0)
Stage-2 risk protocol: {'risk': 0.0025, 'gross_lev': 4.0, 'max_trades': 3, 'daily_loss': 0.0, 'dd_throttle': 0.0, 'eq_filter': True}
Search runtime 180s


## Analyze winning strategy
The "winner" is the configuration the walk-forward process selected for the most recent years. It is read
directly from the search results (printed below), so this section stays correct if the search is re-run.
**Live start** is the first year of that final selection streak. Everything before it was available when the
config was chosen (in-sample); everything from it onward is genuinely out-of-sample.

In [16]:
# ═════════════════════════════════════════════════════════════════════════════
# WINNING ORB CONFIG — WHEN DOES IT WORK, AND WHEN DOES IT FAIL?
# Every trade is tagged with market conditions. Ex-ante conditions are known before entry (usable as
# filters); the EX-POST condition describes the day after the fact (explains results, not tradable).
# Results are in R (risk units, after costs).
# ═════════════════════════════════════════════════════════════════════════════
k_final = int(wf_log["chosen_config"].iloc[-1])
streak = wf_log["chosen_config"].iloc[::-1].eq(k_final).cumprod().astype(bool)
LIVE_START = pd.Timestamp(year=int(wf_log["trade_year"].iloc[::-1][streak].min()), month=1, day=1)
WIN_LOGIC, WIN_FILT = res.iloc[k_final]["_logic"], res.iloc[k_final]["_filt"]
win_trades = apply_filter(candidates(WIN_LOGIC), WIN_FILT).sort_values(["date", "entry_col"]).copy()
MIN_BUCKET_TRADES = 30
print(f"Winner (config #{k_final}, live from {LIVE_START.year}):\n  {WIN_LOGIC}\n  {WIN_FILT}")

# Market context from SPY (ex-ante: shifted so it is known at today's open)
spy = panels["SPY"]["daily"]
spy_ret = spy["close"].pct_change()
spy_tr = pd.concat([spy["high"] - spy["low"], (spy["high"] - spy["close"].shift(1)).abs(),
                    (spy["low"] - spy["close"].shift(1)).abs()], axis=1).max(axis=1)
ctx = pd.DataFrame({
    "spy_vol20": (spy_ret.rolling(20, min_periods=20).std() * np.sqrt(252)).shift(1),
    "spy_above_200d": (spy["close"] > spy["close"].rolling(200, min_periods=200).mean()).shift(1),
    "spy_ret_5d": spy["close"].pct_change(5).shift(1),
    "spy_day_move_atr": (spy["close"] - spy["open"]).abs()
                        / spy_tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean().shift(1),   # ex-post
}, index=spy.index)
tr = win_trades.join(ctx, on="date")


def bucket(s: pd.Series, edges: list, labels: list) -> pd.Series:
    return pd.cut(s, edges, labels=labels, right=False)


minute = tr["entry_col"] + RTH_OPEN
COND = {
    "Volatility regime (SPY 20d)": bucket(tr["spy_vol20"], [0, 0.12, 0.20, 0.30, np.inf],
                                          ["calm <12%", "normal 12–20%", "elevated 20–30%", "crisis >30%"]),
    "Market trend (SPY vs 200d)": tr["spy_above_200d"].map({True: "above 200d", False: "below 200d"}),
    "Prior week (SPY 5d return)": bucket(tr["spy_ret_5d"], [-np.inf, -0.02, 0.0, 0.02, np.inf],
                                         ["≤ −2%", "−2–0%", "0–2%", "≥ +2%"]),
    "ETF": tr["asset"],
    "Direction": tr["dir"].map({1: "long", -1: "short"}),
    "Gap size (|gap| / ATR)": bucket(tr["gap_atr"].abs(), [0, 0.3, 0.5, 1.0, np.inf], ["< 0.3", "0.3–0.5", "0.5–1.0", "> 1.0"]),
    "OR width (OR / ATR)": bucket(tr["or_atr"], [0, 0.2, 0.35, 0.5, np.inf], ["< 0.20", "0.20–0.35", "0.35–0.50", "> 0.50"]),
    "Entry time": bucket(minute, [0, 600, 630, 690, 24 * 60], ["before 10:00", "10:00–10:30", "10:30–11:30", "after 11:30"]),
    "Vol trend (ATR14/ATR50)": pd.Series(np.where(tr["atr_ratio"] > 1, "expanding", "contracting"), index=tr.index),
    "Day of week": pd.Series(pd.Categorical(tr["date"].dt.day_name().str[:3], ["Mon", "Tue", "Wed", "Thu", "Fri"],
                                            ordered=True), index=tr.index),
    "Day type (SPY |close−open| / ATR) — EX-POST": bucket(tr["spy_day_move_atr"], [0, 0.3, 0.7, np.inf],
                                                          ["range < 0.3", "0.3–0.7", "trend > 0.7"]),
}
live = tr["date"] >= LIVE_START


def group_stats(key: pd.Series) -> pd.DataFrame:
    g = tr.groupby(key, observed=True)["R"]
    out = pd.DataFrame({"trades": g.size(), "mean_R": g.mean(), "se": g.std(ddof=1) / np.sqrt(g.size()),
                        "hit_%": 100 * g.apply(lambda s: (s > 0).mean()), "total_R": g.sum()})
    out["t_stat"] = out["mean_R"] / out["se"]
    out["live_mean_R"] = tr.loc[live].groupby(key[live], observed=True)["R"].mean()
    return out


R = tr["R"]
print(f"\n{len(tr):,} trades {tr['date'].min().date()} → {tr['date'].max().date()} | mean {R.mean():+.3f} R "
      f"(t = {R.mean() / (R.std() / np.sqrt(len(R))):.2f}) | hit {100 * (R > 0).mean():.1f}% | "
      f"avg win {R[R > 0].mean():+.2f} R, avg loss {R[R <= 0].mean():+.2f} R | live-period mean {R[live].mean():+.3f} R")

tables = {c: group_stats(k) for c, k in COND.items()}
ranked = (pd.concat([t.assign(condition=c) for c, t in tables.items()]).reset_index(names="bucket")
          .query("trades >= @MIN_BUCKET_TRADES").sort_values("mean_R", ascending=False))
cols = ["condition", "bucket", "trades", "mean_R", "t_stat", "hit_%", "total_R", "live_mean_R"]
print(f"\nBEST conditions (≥ {MIN_BUCKET_TRADES} trades):")
display(ranked.head(6)[cols].round(3))
print("WORST conditions:")
display(ranked.tail(6).iloc[::-1][cols].round(3))
print(f"Caution: {len(ranked)} buckets are compared, so |t| ≈ 2 can occur by chance; treat |t| < 3, or buckets "
      "whose live-period mean disagrees in sign, as unconfirmed.")

# Figure 1 — mean R by condition (bars ± 1 SE)
ncols, nrows = 3, int(np.ceil(len(COND) / 3))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=list(COND), vertical_spacing=0.09, horizontal_spacing=0.06)
for i, c in enumerate(COND):
    t = tables[c]
    rr, cc = divmod(i, ncols)
    color = np.where(t["trades"] < MIN_BUCKET_TRADES, "#bdbdbd", np.where(t["mean_R"] >= 0, "#2ca02c", "#d62728"))
    fig.add_trace(go.Bar(x=t.index.astype(str), y=t["mean_R"], marker_color=color,
                         error_y=dict(type="data", array=t["se"], thickness=1, width=3),
                         text=[f"n={n}" for n in t["trades"]], textposition="outside", textfont=dict(size=9),
                         hovertemplate="%{x}<br>mean R %{y:.3f}<br>%{text}<extra></extra>", showlegend=False),
                  row=rr + 1, col=cc + 1)
    fig.add_hline(y=R.mean(), line_dash="dot", line_color="grey", line_width=1, row=rr + 1, col=cc + 1)
    fig.update_yaxes(title_text="mean R" if cc == 0 else None, zeroline=True, zerolinecolor="black", row=rr + 1, col=cc + 1)
fig.update_annotations(font_size=11)
fig.update_layout(title=("Winning ORB config — average result per trade by market condition"
                         "<br><sup>Green = profitable, red = losing, grey = < 30 trades · error bars ±1 SE · "
                         "dotted line = overall average · EX-POST panel is explanatory only</sup>"),
                  template="plotly_white", height=330 * nrows, bargap=0.35)
fig.show()

# Figure 2 — cumulative R with the 3 worst drawdowns and the best 12 months highlighted
cumR = tr.groupby("date")["R"].sum().reindex(CAL, fill_value=0.0).cumsum()
cumR = cumR.loc[cumR.index >= tr["date"].min()]
dd = cumR - cumR.cummax()
episodes, start = [], None
for dt, v in dd.items():
    if v < 0 and start is None:
        start = dt
    elif v == 0 and start is not None:
        episodes.append((cumR.loc[:start].idxmax(), dd.loc[start:dt].idxmin(), dd.loc[start:dt].min()))
        start = None
if start is not None:
    episodes.append((cumR.loc[:start].idxmax(), dd.loc[start:].idxmin(), dd.loc[start:].min()))
roll = cumR.diff(252)
best_end = roll.idxmax()
best_start = cumR.index[max(cumR.index.get_loc(best_end) - 252, 0)]
fig = go.Figure(go.Scatter(x=cumR.index, y=cumR.values, line=dict(color="#1f77b4", width=2), name="cumulative R"))
for k, (pk, trough, depth) in enumerate(sorted(episodes, key=lambda e: e[2])[:3], 1):
    fig.add_vrect(x0=pk, x1=trough, fillcolor="#d62728", opacity=0.15, line_width=0)
    fig.add_annotation(x=trough, y=cumR.loc[trough], text=f"DD #{k}: {depth:.1f} R<br>{pk:%b %Y}–{trough:%b %Y}",
                       showarrow=True, arrowhead=2, ax=0, ay=40, font=dict(size=10, color="#d62728"))
fig.add_vrect(x0=best_start, x1=best_end, fillcolor="#2ca02c", opacity=0.15, line_width=0)
fig.add_annotation(x=best_end, y=cumR.loc[best_end], text=f"best 12 months: +{roll.max():.1f} R", showarrow=True,
                   arrowhead=2, ax=0, ay=-40, font=dict(size=10, color="#2ca02c"))
fig.add_vline(x=LIVE_START, line_dash="dash", line_color="grey")
fig.add_annotation(x=LIVE_START, y=1, yref="y domain", text="  live →", showarrow=False, xanchor="left")
fig.update_layout(title="Winning ORB config — cumulative R (red = 3 worst drawdowns, green = best 12 months)",
                  yaxis_title="cumulative R (after costs)", template="plotly_white", height=480, showlegend=False)
fig.show()

Winner (config #9323, live from 2017):
  Logic(n=15, dir_mode='candle+gap', buffer=0.02, confirm='close', cutoff=120, stop=('or_mid', 0.0), be_R=2.0, trail_atr=0.0, target_R=0.0, exit_col=270)
  Filt(rv_min=0.0, or_atr_max=0.5, gap_min=0.3, regime='any', nr7=False, universe='equity', top_n=None, min_avg_vol=0.0, min_atr=0.0)

2,844 trades 2003-01-24 → 2026-09-21 | mean +0.227 R (t = 7.46) | hit 42.2% | avg win +1.79 R, avg loss -0.92 R | live-period mean +0.255 R

BEST conditions (≥ 30 trades):


,condition,bucket,trades,mean_R,t_stat,hit_%,total_R,live_mean_R
35,Day type (SPY |close−open| / ATR) — EX-POST,trend > 0.7,774,1.089,14.833,61.111,843.218,1.208
3,Volatility regime (SPY 20d),crisis >30%,175,0.494,3.834,50.286,86.411,0.264
18,Gap size (|gap| / ATR),> 1.0,153,0.413,2.808,43.137,63.122,0.718
2,Volatility regime (SPY 20d),elevated 20–30%,445,0.374,4.700,45.618,166.352,0.484
5,Market trend (SPY vs 200d),below 200d,701,0.366,5.717,45.364,256.806,0.326
32,Day of week,Fri,515,0.325,4.379,44.272,167.509,0.429


WORST conditions:


,condition,bucket,trades,mean_R,t_stat,hit_%,total_R,live_mean_R
33,Day type (SPY |close−open| / ATR) — EX-POST,range < 0.3,962,-0.431,-12.759,22.869,-414.605,-0.407
24,Entry time,10:30–11:30,370,0.102,1.593,47.297,37.790,0.219
21,OR width (OR / ATR),0.35–0.50,659,0.103,1.935,43.096,68.021,0.084
10,ETF,DIA,696,0.166,2.842,41.523,115.365,0.074
29,Day of week,Tue,603,0.169,2.682,41.459,101.835,0.229
1,Volatility regime (SPY 20d),normal 12–20%,1003,0.171,3.512,41.176,171.132,0.188


Caution: 35 buckets are compared, so |t| ≈ 2 can occur by chance; treat |t| < 3, or buckets whose live-period mean disagrees in sign, as unconfirmed.


#### **Where it does best**
Trend days, by a wide margin. On days when SPY closes far from its open, the average is +1.09R per trade (+1.21R in the live period, 2017+). On quiet, range-bound days it’s −0.43R. That label is only known after the fact, so it can’t be used as a filter. It does tell you what the strategy is: a bet that today will trend, and almost everything below is a proxy for that.
High volatility. When SPY’s 20-day volatility is above 20 %, trades average +0.37 to +0.49R, against about +0.17R in calm markets. The best single stretch was Oct 2007 – Oct 2008, at +121R.
Markets below their 200-day average: +0.37R vs +0.18R above it.
Bigger gaps and very narrow ranges. Gaps above 1 × ATR average +0.41R. Opening ranges under 0.35 × ATR do better than wider ones.
SPY and QQQ: +0.26R each, versus +0.17R for DIA.
Long and short are about equal (+0.23R vs +0.22R), so it isn’t just riding the bull market.

#### **Where it does worst**
Calm, rising markets. Volatility under 12 % with SPY above its 200-day average is the most common condition (1,203 trades), and the edge is thinnest there: about +0.18R.
Late entries (10:30–11:30), wide ranges (0.35–0.5 × ATR) and DIA. All three are positive but weak, with t-statistics below 3. Wide ranges and DIA nearly vanish in the live period (+0.08R and +0.07R).
The worst periods were 2004–05, mid-2023 to early 2024, and Aug 2025 to Apr 2026, each a drawdown of about 30R. Two of the three worst drawdowns are recent, and 2025 was roughly flat (+1R). That could be a low-volatility patch or the edge fading, and it’s too early to tell which.
#### **What this suggests**

*Two filters are worth testing next: skip entries after 10:30, and skip opening ranges wider than 0.35 × ATR. Both have a clear rationale, and both weak buckets also stayed weak in the live period. Adding them now would be fitting to the same data, though, so they would need confirming on new data.*

Combined conditions are thin at the extremes: crisis volatility while SPY is above its 200-day average has 1 trade, and calm volatility while SPY is below it has 13. Don't read anything into those combinations.

## Winner vs SPY buy & hold
Two realistic implementations of the winner against SPY:
* **ORB @ 1 % risk.** Integer shares, 1 % of equity risked per trade, 4× gross-leverage cap.
* **70 % SPY / 30 % ORB.** A buy-and-hold core with an ORB sleeve, rebalanced daily.

The live period (from `LIVE_START`) is genuinely out-of-sample.

In [17]:
CRISES = {"GFC 2007–09": ("2007-10-09", "2009-03-09"), "Aug 2011 downgrade": ("2011-07-22", "2011-10-03"),
          "Q4 2018 selloff": ("2018-09-20", "2018-12-24"), "COVID crash 2020": ("2020-02-19", "2020-03-23"),
          "2022 bear market": ("2022-01-03", "2022-10-12"), "2025 tariff shock": ("2025-02-19", "2025-04-08")}
BLEND_SPY = 0.70
ORB_NAME, BLEND_NAME, SPY_NAME = "ORB @ 1% risk (4× cap)", f"{BLEND_SPY:.0%} SPY / {1 - BLEND_SPY:.0%} ORB", f"{BENCHMARK} buy & hold"

cmp = pd.DataFrame(index=CAL)
cmp[SPY_NAME] = R_SPY
cmp[ORB_NAME] = run_risk(win_trades, dict(risk=0.01, gross_lev=4.0, max_trades=99, daily_loss=0.0,
                                          dd_throttle=0.0, eq_filter=False))
cmp[BLEND_NAME] = BLEND_SPY * cmp[SPY_NAME] + (1 - BLEND_SPY) * cmp[ORB_NAME]
cmp = cmp.loc[cmp.index >= win_trades["date"].min()]
COLORS = {SPY_NAME: "#d62728", ORB_NAME: "#1f77b4", BLEND_NAME: "#2ca02c"}

for label, m in ((f"FULL PERIOD (partly in-sample before {LIVE_START.year})", cmp.index >= cmp.index[0]),
                 (f"LIVE / OUT-OF-SAMPLE ({LIVE_START.year}+)", cmp.index >= LIVE_START)):
    sub = cmp.loc[m]
    print(f"\n{label}: {sub.index[0].date()} → {sub.index[-1].date()}")
    display(pd.DataFrame({c: perf_stats(sub[c], sub[SPY_NAME]) for c in sub}).round(2))
print("\nReturn during SPY's major drawdowns (%):")
display(pd.DataFrame({name: {c: 100 * ((1 + cmp.loc[a:b, c]).prod() - 1) for c in cmp}
                      for name, (a, b) in CRISES.items()}).T.round(1))

# Figure 1 — growth, drawdown, rolling 12-month return, rolling correlation
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[0.40, 0.20, 0.22, 0.18], vertical_spacing=0.035,
                    subplot_titles=("Growth of $25,000 (log scale)", "Drawdown (%)", "Rolling 12-month return (%)",
                                    "Rolling 12-month correlation of ORB to SPY"))
for c in cmp:
    eq = INITIAL_EQUITY * (1 + cmp[c]).cumprod()
    fig.add_trace(go.Scatter(x=eq.index, y=eq, name=c, line=dict(color=COLORS[c], width=2.5), legendgroup=c), 1, 1)
    fig.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.cummax() - 1), name=c, showlegend=False,
                             line=dict(color=COLORS[c], width=1.2), legendgroup=c), 2, 1)
    fig.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.shift(252) - 1), name=c, showlegend=False,
                             line=dict(color=COLORS[c], width=1.2), legendgroup=c), 3, 1)
fig.add_trace(go.Scatter(x=cmp.index, y=cmp[ORB_NAME].rolling(252).corr(cmp[SPY_NAME]), showlegend=False,
                         line=dict(color="#555555", width=1.2)), 4, 1)
for row in range(1, 5):
    fig.add_vrect(x0=LIVE_START, x1=cmp.index[-1], fillcolor="grey", opacity=0.07, line_width=0, row=row, col=1)
    for a, b in CRISES.values():
        fig.add_vrect(x0=a, x1=b, fillcolor="#d62728", opacity=0.06, line_width=0, row=row, col=1)
for row in (3, 4):
    fig.add_hline(y=0, line_color="black", line_width=0.8, row=row, col=1)
fig.add_annotation(x=LIVE_START, y=1, yref="y domain", text="  live / out-of-sample →", showarrow=False,
                   xanchor="left", row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_yaxes(range=[-1, 1], row=4, col=1)
fig.update_layout(title="Winning ORB strategy vs SPY buy & hold<br><sup>Grey = live / out-of-sample · light red = SPY crisis windows</sup>",
                  template="plotly_white", height=1100, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
fig.show()

# Figure 2 — calendar-year returns side by side + monthly scatter
yearly = (1 + cmp).resample("YE").prod() - 1
yearly.index = yearly.index.year
monthly = (1 + cmp).resample("ME").prod() - 1
fig = make_subplots(rows=1, cols=2, column_widths=[0.64, 0.36], horizontal_spacing=0.08,
                    subplot_titles=("Calendar-year return (%)", "Monthly returns: ORB vs SPY (%)"))
for c in cmp:
    fig.add_trace(go.Bar(x=yearly.index, y=100 * yearly[c], name=c, marker_color=COLORS[c]), 1, 1)
fig.add_trace(go.Scatter(x=100 * monthly[SPY_NAME], y=100 * monthly[ORB_NAME], mode="markers", showlegend=False,
                         marker=dict(size=6, opacity=0.55, color=np.where(monthly.index >= LIVE_START, "#1f77b4", "#9ecae1")),
                         text=monthly.index.strftime("%b %Y"), hovertemplate="%{text}<br>SPY %{x:.1f}%<br>ORB %{y:.1f}%"), 1, 2)
fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=2)
fig.add_vline(x=0, line_color="black", line_width=0.8, row=1, col=2)
fig.update_xaxes(title_text="SPY monthly return %", row=1, col=2)
fig.update_yaxes(title_text="ORB monthly return %", row=1, col=2)
fig.update_layout(template="plotly_white", height=520, barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="left", x=0),
                  title=f"Year-by-year and month-by-month comparison (dark dots = live period, {LIVE_START.year}+)")
fig.show()


FULL PERIOD (partly in-sample before 2017): 2003-01-24 → 2026-09-21


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
Total return %,1144.80,2372.17,1658.91
CAGR %,11.27,14.55,12.91
Volatility %,18.49,13.90,13.36
Sharpe,0.67,1.05,0.98
Sortino,0.94,2.31,1.47
Max drawdown %,-55.20,-30.82,-28.81
Calmar,0.20,0.47,0.45
Recovery (sessions),869.00,550.00,96.00
Worst day %,-11.63,-3.81,-8.14
Worst month %,-16.87,-6.34,-7.96



LIVE / OUT-OF-SAMPLE (2017+): 2017-01-03 → 2026-09-21


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
Total return %,276.93,547.47,371.05
CAGR %,14.67,21.26,17.34
Volatility %,18.25,15.35,13.34
Sharpe,0.84,1.33,1.27
Sortino,1.17,3.15,1.91
Max drawdown %,-33.85,-11.88,-24.59
Calmar,0.43,1.79,0.71
Recovery (sessions),97.00,100.00,70.00
Worst day %,-11.63,-2.59,-8.14
Worst month %,-12.47,-6.34,-7.96



Return during SPY's major drawdowns (%):


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
GFC 2007–09,-54.8,121.2,-24.9
Aug 2011 downgrade,-17.9,-2.4,-13.3
Q4 2018 selloff,-18.5,6.2,-11.6
COVID crash 2020,-33.5,0.4,-24.2
2022 bear market,-28.4,38.4,-11.6
2025 tariff shock,-18.4,-3.4,-14.0


## Machine Learning implementation
utilizes forest tree classification to determine whether the strategy should or should not trade or double down on sizing.